In [2]:
import os
import xarray as xr
import rioxarray
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterio import features
from affine import Affine
from dask.distributed import Client

In [3]:
# Define paths
BASE_DIR = Path("..").resolve()
DATA_DIR = os.path.join("..", "data", "ethiopia")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

### Start dask cluster

In [4]:
# Create a local dask client
client = Client(n_workers=4, threads_per_worker=2, memory_limit='2GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 7.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50661,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:50681,Total threads: 2
Dashboard: http://127.0.0.1:50688/status,Memory: 1.86 GiB
Nanny: tcp://127.0.0.1:50664,


# Crop Cover Extraction for Woredas

This notebook provides functions to extract crop cover data for Ethiopian woredas using xarray and rioxarray.

For processing multiple woredas efficiently, we can rasterize the entire woreda GeoDataFrame into a grid that matches the crop cover data. This allows vectorized operations across all woredas.

In [5]:
def rasterize_woredas(woredas_gdf, crop_cover_template):
    """
    Rasterize woredas into a grid matching the crop cover template.
    
    Parameters
    ----------
    woredas_gdf : gpd.GeoDataFrame
        GeoDataFrame with woreda geometries
    crop_cover_template : xr.DataArray
        Template xarray to match coordinates/shape (e.g., the crop cover VRT)
    
    Returns
    -------
    woreda_grid : xr.DataArray
        Array where each pixel value = woreda index (0 = no woreda)
    woreda_lookup : gpd.GeoDataFrame
        Lookup table mapping woreda_id to NAME_3 and geometry
    """
    # Add a numeric ID column if it doesn't exist
    # Using the index as the woreda ID
    woredas_gdf = woredas_gdf.copy()
    woredas_gdf['woreda_id'] = woredas_gdf.index + 1  # +1 so 0 can be "no woreda"
    
    # Get the spatial extent from the template
    x_coords = crop_cover_template.x.values
    y_coords = crop_cover_template.y.values
    
    # Calculate pixel size
    x_res = abs(x_coords[1] - x_coords[0])
    y_res = abs(y_coords[1] - y_coords[0])
    
    # Create affine transform
    # Affine maps from pixel coordinates to geographic coordinates
    transform = Affine.translation(x_coords[0] - x_res/2, y_coords[0] + y_res/2) * \
                Affine.scale(x_res, -y_res)
    
    # Get shape from template
    height, width = len(y_coords), len(x_coords)
    
    # Prepare (geometry, value) pairs for rasterization
    shapes = ((geom, value) for geom, value in 
              zip(woredas_gdf.geometry, woredas_gdf.woreda_id))
    
    print(f"Rasterizing {len(woredas_gdf)} woredas into {height} x {width} grid...")
    
    # Rasterize using rasterio
    woreda_array = features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,  # Background value (no woreda)
        dtype='uint16',  # Can handle up to 65535 woredas
        all_touched=False  # Only pixels whose center is in polygon
    )
    
    print(f"Rasterization complete!")
    
    # Convert to xarray DataArray with matching coordinates
    woreda_grid = xr.DataArray(
        woreda_array,
        coords={
            'y': y_coords,
            'x': x_coords
        },
        dims=['y', 'x'],
        name='woreda_id'
    )
    
    # Add metadata
    woreda_grid.attrs['description'] = 'Woreda ID grid (0 = no woreda)'
    woreda_grid.attrs['woreda_count'] = len(woredas_gdf)
    
    # Create lookup table
    woreda_lookup = woredas_gdf[['NAME_3', 'woreda_id', 'geometry']].copy()
    
    return woreda_grid, woreda_lookup

### Create the Woreda Grid

This is a one-time operation that creates a rasterized version of all woredas:

In [6]:
# Load crop cover as template
crop_cover_fp = os.path.join(DATA_DIR, "crop_cover", "crop_cover.vrt")
woredas_fp = os.path.join(DATA_DIR, "woredas.json")
crop_cover = rioxarray.open_rasterio(crop_cover_fp, chunks="auto")
if 'band' in crop_cover.dims:
    crop_cover = crop_cover.squeeze('band', drop=True)

# Load woredas
woredas_gdf = gpd.read_file(woredas_fp)

# Rasterize all woredas
woreda_grid, woreda_lookup = rasterize_woredas(woredas_gdf, crop_cover)

print(f"\nWoreda grid shape: {woreda_grid.shape}")
print(f"Number of woredas: {len(woreda_lookup)}")
print(f"Woreda IDs range: {woreda_grid.min().values} to {woreda_grid.max().values}")

Rasterizing 690 woredas into 74482 x 74222 grid...
Rasterization complete!

Woreda grid shape: (74482, 74222)
Number of woredas: 690
Woreda IDs range: 0 to 690


### Save Woreda Grid to Zarr (for reuse)

In [13]:
woreda_grid

<xarray.DataArray 'woreda_id' (y: 74482, x: 74222)> Size: 11GB
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(74482, 74222), dtype=uint16)
Coordinates:
  * y        (y) float64 596kB 20.07 20.07 20.07 ... -0.0006737 -0.0009432
  * x        (x) float64 594kB 30.0 30.0 30.0 30.0 30.0 ... 50.0 50.0 50.0 50.0
Attributes:
    description:   Woreda ID grid (0 = no woreda)
    woreda_count:  690

In [7]:
# Save woreda grid for future use
woreda_grid.to_zarr(os.path.join(OUTPUT_DIR, "woredas.zarr"), mode='w')

# Save lookup table
woreda_lookup.to_file(os.path.join(OUTPUT_DIR, "woredas_lookup.json"), driver='GeoJSON')

print(f"✓ Woreda grid saved")
print(f"✓ Lookup table saved")

c:\Users\geo1k\.local\share\mamba\envs\geodask\Lib\site-packages\zarr\api\asynchronous.py:228: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


✓ Woreda grid saved
✓ Lookup table saved


### Save Woreda Grid to GeoTIFF

In [15]:
# Save woreda_grid to GeoTIFF with EPSG:4326
WOREDA_GRID_GEOTIFF = os.path.join(OUTPUT_DIR, "woredas.tif")

# Set the CRS to EPSG:4326
woreda_grid.rio.write_crs("EPSG:4326", inplace=True)

# Save to GeoTIFF with DEFLATE compression
woreda_grid.rio.to_raster(
    WOREDA_GRID_GEOTIFF,
    driver='GTiff',
    compress='DEFLATE',  # DEFLATE compression to reduce file size
    dtype='uint16'       # Matches the data type (can handle up to 65535 woredas)
)

print(f"✓ Woreda grid saved to {WOREDA_GRID_GEOTIFF}")

✓ Woreda grid saved to ..\data\ethiopia\output\woredas.tif


### Use Woreda Grid to Extract Specific Woreda

In [14]:
# Extract Kobo using the woreda grid
kobo_id = woreda_lookup[woreda_lookup['NAME_3'] == 'Kobo']['woreda_id'].values[0]
print(f"Kobo woreda_id: {kobo_id}")

# Create mask for Kobo
kobo_mask = woreda_grid == kobo_id

# Extract crop cover for Kobo
kobo_crop_from_grid = crop_cover.where(kobo_mask, drop=True)
kobo_crop_binary = (kobo_crop_from_grid == 2).astype('uint8')

print(f"\nKobo crop cover (from grid):")
print(f"  Shape: {kobo_crop_binary.shape}")
print(f"  Crop percentage: {(kobo_crop_binary.sum() / kobo_crop_binary.size * 100).values:.2f}%")

Kobo woreda_id: 162

Kobo crop cover (from grid):
  Shape: (1734, 1909)
  Crop percentage: 16.92%


### Batch Process ALL Woredas at Once

This is where the woreda grid approach really shines - you can compute statistics for all woredas in a single vectorized operation:

In [ ]:
# Convert crop cover to binary (1=crop, 0=no crop)
crop_binary = (crop_cover == 2).astype('uint8')

# Group by woreda and calculate statistics
print("Computing crop statistics for all woredas...")

# Total crop pixels per woreda
crop_pixels_per_woreda = crop_binary.groupby(woreda_grid).sum()

# Total pixels per woreda
total_pixels_per_woreda = woreda_grid.groupby(woreda_grid).count()

# Crop coverage percentage
crop_percentage_per_woreda = (crop_pixels_per_woreda / total_pixels_per_woreda * 100)

print("✓ Statistics computed!")

Computing crop statistics for all woredas...


In [ ]:
# Display results for a few woredas
print("\nSample crop coverage statistics:")
print("-" * 50)

for idx in [162, 163, 164]:  # Kobo is 162
    if idx in woreda_lookup['woreda_id'].values:
        woreda_name = woreda_lookup[woreda_lookup['woreda_id'] == idx]['NAME_3'].values[0]
        
        # Note: groupby results are indexed by the group value (woreda_id)
        # We need to select the appropriate value
        crop_pct = crop_percentage_per_woreda.sel(woreda_id=idx).values
        total_px = total_pixels_per_woreda.sel(woreda_id=idx).values
        crop_px = crop_pixels_per_woreda.sel(woreda_id=idx).values
        
        print(f"{woreda_name:20} (ID {idx}): {crop_pct:.2f}% crop ({crop_px:,}/{total_px:,} pixels)")

### Create a DataFrame with All Woreda Statistics

In [ ]:
import pandas as pd

# Create a comprehensive DataFrame with all woreda statistics
results = []

for idx in woreda_lookup['woreda_id'].values:
    woreda_name = woreda_lookup[woreda_lookup['woreda_id'] == idx]['NAME_3'].values[0]
    
    crop_pct = crop_percentage_per_woreda.sel(woreda_id=idx).values
    total_px = total_pixels_per_woreda.sel(woreda_id=idx).values
    crop_px = crop_pixels_per_woreda.sel(woreda_id=idx).values
    
    results.append({
        'woreda_id': idx,
        'woreda_name': woreda_name,
        'total_pixels': int(total_px),
        'crop_pixels': int(crop_px),
        'crop_percentage': float(crop_pct)
    })

woreda_stats_df = pd.DataFrame(results)

# Display top 10 woredas by crop coverage
print("Top 10 woredas by crop coverage:")
print(woreda_stats_df.nlargest(10, 'crop_percentage')[['woreda_name', 'crop_percentage', 'crop_pixels']])

In [ ]:
# Save statistics to CSV
STATS_CSV_PATH = OUTPUT_DIR / "woreda_crop_statistics.csv"
woreda_stats_df.to_csv(STATS_CSV_PATH, index=False)

print(f"\n✓ Statistics saved to {STATS_CSV_PATH}")

In [37]:
woreda_name = "Kobo"
woredas_gdf = gpd.read_file(WOREDAS_PATH)
woreda = woredas_gdf[woredas_gdf['NAME_3'] == woreda_name]
woreda

,GID_3,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,GID_2,NAME_2,NL_NAME_2,NAME_3,VARNAME_3,NL_NAME_3,TYPE_3,ENGTYPE_3,CC_3,HASC_3,geometry
161,ETH.3.11.7_1,ETH,Ethiopia,ETH.3_1,Amhara,NA,ETH.3.11_1,SemenWello,NA,Kobo,NA,NA,Woreda,District,030302,NA,"MULTIPOLYGON (((39.8834 11.9266, 39.8871 11.90..."


In [16]:
client.shutdown()